# Grow until the leaves are pure

The root question is length $\le 7/2$.

- Right child: lengths $5$ and $5$, both fail. Gini $0$. Stop.
- Left child: four pass and two fail. Gini $4/9$. Ask another question.

Inside the left child the lengths are only $1$ and $2$. Cutting at $3/2$ gives two groups of three, each with two pass and one fail. Both children have Gini $4/9$, the same as the parent, so the gain is $0$.

Weight still has a gap. Weight $\le 3$ puts four passes on the left and two fails on the right. Both sides are pure, so the weighted Gini is $0$ and the gain is the whole $4/9$.

The grown tree:

1. Length $\le 7/2$?
2. If yes, weight $\le 3$? Pass on the left, fail on the right.
3. If no, fail.

Every training part lands in a pure leaf. Training error is $0$.


In [1]:
# Second split: length inside the left child gains 0; weight gains 4/9.
from fractions import Fraction

left_child = [
    (1, 1, "pass"), (1, 2, "pass"), (2, 1, "pass"), (2, 2, "pass"),
    (1, 4, "fail"), (2, 4, "fail"),
]

def gini(labels):
    n = len(labels)
    counts = {}
    for label in labels:
        counts[label] = counts.get(label, 0) + 1
    return 1 - sum(Fraction(c, n) ** 2 for c in counts.values())

parent = gini([row[2] for row in left_child])
assert parent == Fraction(4, 9)

def gain(feature, threshold):
    left = [row[2] for row in left_child if row[feature] <= threshold]
    right = [row[2] for row in left_child if row[feature] > threshold]
    weighted = (
        Fraction(len(left), 6) * gini(left) + Fraction(len(right), 6) * gini(right)
    )
    return parent - weighted

assert gain(0, Fraction(3, 2)) == 0
assert gain(1, 3) == Fraction(4, 9)
print("length gain", gain(0, Fraction(3, 2)))
print("weight gain", gain(1, 3))


length gain 0
weight gain 4/9


In [2]:
# Every training part reaches a pure leaf. A short heavy part fails.
from fractions import Fraction

parts = [
    (1, 1, "pass"), (1, 2, "pass"), (2, 1, "pass"), (2, 2, "pass"),
    (1, 4, "fail"), (2, 4, "fail"), (5, 1, "fail"), (5, 2, "fail"),
]

def predict(length, weight):
    if length <= Fraction(7, 2):
        if weight <= 3:
            return "pass"
        return "fail"
    return "fail"

for length, weight, label in parts:
    assert predict(length, weight) == label
assert predict(2, 4) == "fail"
assert predict(2, 2) == "pass"
print("training mistakes", 0)


training mistakes 0


## When to stop

Stop when the leaf is pure, or when every available cut has gain $0$, or when the leaf holds one part. On this table, purity arrives first.

A cut with gain $0$ does not have to be useless two questions later. The rectangle lesson will show XOR, where the first cut gains $0$ and the second cut finishes the job. The rule "stop when the best gain is $0$" misses that pattern. It is still the rule used here, and it is the right rule to state out loud.


## A pitfall

A tree with training error $0$ has finished the training table. It has not finished the next part. Part $(5, 4)$ is long, so this tree says fail, even though the table never contained a long heavy part. The right leaf borrowed its label from the long light parts.

## What to carry forward

Grow by repeating the gain calculation. Here the root is length $\le 7/2$ with gain $1/6$, and the only impure child then asks weight $\le 3$ with gain $4/9$. All eight training labels are recovered.
